# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aisyahnabillah/ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Ranked actions, in plain words: pages are flagged into two reason codes based on two validated signals.

review_for_refresh (reason: stale_but_visible) uses the Week-4 baseline rule, validated in ML-08/09 as outperforming my trained models on this data and split (precision@20 = 0.65 vs 0.45 for Random Forest on a client-grouped holdout).

review_snippet_or_metadata (reason: low_ctr_for_position) uses Signal 2 from ML-07, CONFIRMED: CTR drops consistently as position tier worsens (top_3: 2.71% -> beyond: 0.21%). A page is flagged if its CTR is below HALF its own position tier's average, so a page is only flagged if it is clearly underperforming its direct peers at the same visibility level, not just below an average.

Archetype -> action mapping:
- Stale but visible -> review_for_refresh (content likely needs updating)
- Visible, but CTR well below its position tier's peers -> review_snippet_or_metadata 
  (likely a title/meta/snippet problem, not a content-freshness problem)
- Everything else -> monitor (no action needed right now)

Decay/refresh insight: the two reason codes point to different root causes that can look 
similar on the surface (both are "visible pages that could improve"). A refresh will not fix 
a snippet problem, and a snippet rewrite will not fix stale content, separating them prevents 
the wrong fix being applied to the wrong page.

Design correction: my first version combined both reason codes into one numeric score, but 
this caused a scale problem. low_ctr_for_position was flagged on 45.5% of all pages (13,665 
of 30,000) using a "below tier average" threshold, which is really just a median split, not a 
meaningful signal. Combined with impressions_90d as the scoring weight, this reason code 
completely dominated the ranking, and stale_but_visible (only 17 pages, but the better-
validated signal per ML-08/09) never appeared in the top 20 at all.

Fix: (1) tightened the CTR threshold to below HALF the tier average (9,752 pages flagged, a 
more meaningful gap), and (2) split the output into two separate sub-queues by reason code 
instead of one combined ranking, since a refresh action and a snippet-review action are 
different reviewer workflows, not values that should compete on the same numeric scale.

Export decision: both reason codes are exported together in one file, but ranked SEPARATELY 
within their own reason_code (rank_within_reason), not merged into one combined score. A 
content team lead can filter by reason_code to get the queue relevant to their team, rather 
than one arbitrary blended ranking.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Reason code 1: stale + visible (from ML-07, validated as the best-performing approach in ML-08/09)
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

# Reason code 2: CTR underperforming for its position tier (Signal 2 from ML-07, CONFIRMED pattern)
df["position_tier"] = pd.cut(df["avg_position"], bins=[0, 3, 10, 20, 100],
                               labels=["top_3", "page_1", "page_2", "beyond"])
tier_avg_ctr = df.groupby("position_tier", observed=True)["ctr"].transform("mean")
df["low_ctr_for_tier"] = ((df["ctr"] < 0.5 * tier_avg_ctr) & (df["impressions_90d"] >= 500)).astype(int)

print(f"Low CTR flagged (tightened): {df['low_ctr_for_tier'].sum()} out of {len(df)}")


def assign_action(row):
    if row["stale"] == 1 and row["visible"] == 1:
        return "review_for_refresh", "stale_but_visible"
    elif row["low_ctr_for_tier"] == 1:
        return "review_snippet_or_metadata", "low_ctr_for_position"
    else:
        return "monitor", "not_flagged"

df[["action_label", "reason_code"]] = df.apply(lambda r: pd.Series(assign_action(r)), axis=1)

refresh_queue = df[df["reason_code"] == "stale_but_visible"].sort_values("impressions_90d", ascending=False)
snippet_queue = df[df["reason_code"] == "low_ctr_for_position"].sort_values("impressions_90d", ascending=False)

print("Top 10 — review_for_refresh:")
print(refresh_queue[["content_id", "impressions_90d", "days_since_last_update", "reason_code"]].head(10))

print("\nTop 10 — review_snippet_or_metadata:")
print(snippet_queue[["content_id", "impressions_90d", "avg_position", "ctr", "reason_code"]].head(10))

Low CTR flagged (tightened): 9752 out of 30000
Top 10 — review_for_refresh:
                 content_id  impressions_90d  days_since_last_update  \
16751  content_cf56e2e2e282            61678                     194   
16514  content_7368877ea310            59472                     194   
7021   content_1bfaa38ff26c            25715                     194   
21268  content_0a91db491d14            13299                     193   
11489  content_5feee3994adb             7812                     194   
12045  content_c2d929d83eaa             7558                     193   
698    content_b16bd7307b39             4590                     194   
5327   content_fe16a55cd13d             4556                     194   
26810  content_ecb6215e79fd             4429                     194   
20837  content_928af3e22c80             1697                     193   

             reason_code  
16751  stale_but_visible  
16514  stale_but_visible  
7021   stale_but_visible  
21268  stale_but_visibl

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: this playbook is decision-support for a content reviewer with limited time. It 
ranks candidates worth looking at first, it does not replace human judgment on any individual 
page.

Limits:
- Based on a single anonymized starter slice (30,000 rows, one snapshot), not the full warehouse.
- The staleness signal itself showed an OPPOSITE-direction result in ML-07's signal check on 
  this data (stale pages had a LOWER decline rate, though on a small n=174 sample), so 
  stale_but_visible should be read as "worth a look", not as "confirmed declining".
- The Random Forest model tested in ML-08/09 scored below the base rate on a client-grouped 
  holdout, so it is intentionally NOT used as the basis for this playbook, only the simpler, 
  better-validated baseline rule and the CONFIRMED CTR-position signal are used.
- This is observational, cross-sectional data. Nothing here is a causal claim, "review this 
  page" is decision-support language, not a guarantee a refresh or snippet edit will improve 
  performance.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any flagged page, a human should check:
- Is this page already scheduled for consolidation or removal? (the rule can't know that)
- Does the low_ctr_for_position flag actually point to a snippet issue, or could the query 
  intent itself have shifted?
- For refresh candidates with an already-strong position (e.g. avg_position under 10), confirm 
  a refresh is actually the right lever before spending review time, ML-08's error analysis 
  found the model over-flagged well-ranked pages as "at risk".

What should NEVER be automated:
- Auto-publishing content or snippet changes without human review, this playbook ranks 
  candidates, it does not write or approve content.
- Auto-deprioritizing (removing from review) any page purely because it wasn't flagged, 
  absence of a flag is not evidence the page is fine, it just means it didn't match these two 
  specific patterns.
- Treating action_score, rank_within_reason, or any number here as a performance guarantee in 
  any client-facing communication.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Signals that this playbook has gone stale and needs review:
- The stale_but_visible flag rate drifts far from what ML-07 measured (17 out of 30,000), a 
  large shift suggests the underlying content/update patterns have changed.
- The low_ctr_for_position flag rate drifts far from 9,752 out of 30,000 (about 32%), either 
  direction suggests the tier-average baseline itself has shifted and needs recomputing.
- Precision@20 against real reviewer outcomes drops meaningfully below the 0.65 baseline 
  figure measured in ML-08, if reviewers stop finding flagged pages useful, the rule needs 
  re-checking.
- A new data snapshot becomes available (later month, or full warehouse), both rules should be 
  re-validated on it before being trusted again, per the lane guide's warning that client 
  history and seasonality are highly uneven.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

refresh_queue["rank_within_reason"] = range(1, len(refresh_queue) + 1)
snippet_queue["rank_within_reason"] = range(1, len(snippet_queue) + 1)

playbook_queue = pd.concat([refresh_queue, snippet_queue], ignore_index=True)
playbook_queue.to_csv("../outputs/content_action_playbook.csv", index=False)

import json
metrics = {
    "baseline_precision_at_20": 0.65,
    "baseline_precision_at_50": 0.64,
    "rf_grouped_precision_at_20": 0.450,
    "rf_grouped_base_rate": 0.559,
    "n_stale_visible_flagged": int((df["reason_code"] == "stale_but_visible").sum()),
    "n_low_ctr_flagged_tightened": int(df["low_ctr_for_tier"].sum()),
    "low_ctr_threshold": "ctr < 0.5x tier average ctr",
    "n_total": len(df),
}
with open("../outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(metrics)
print(f"\nExported {len(playbook_queue)} total flagged rows across both reason codes")

{'baseline_precision_at_20': 0.65, 'baseline_precision_at_50': 0.64, 'rf_grouped_precision_at_20': 0.45, 'rf_grouped_base_rate': 0.559, 'n_stale_visible_flagged': 17, 'n_low_ctr_flagged_tightened': 9752, 'low_ctr_threshold': 'ctr < 0.5x tier average ctr', 'n_total': 30000}

Exported 9762 total flagged rows across both reason codes


Exported work/outputs/content_action_playbook.csv (the combined ranked queue across both 
reason codes, regenerated each run, not committed per the CI leak-guard) and 
work/outputs/playbook_metrics.json (committed, the receipts this playbook's numbers trace 
back to: baseline vs model precision@20, base rate, and flagged counts for each reason code, 
reused directly in next week's paper).

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.